# PVT Study Starter

**Task:** [Replace with task title]
**Date:** [YYYY-MM-DD]

Template for PVT laboratory simulations: CME, CVD, differential liberation, saturation pressure, and GOR.

In [ ]:
# ── NeqSim Setup (dual-boot: devtools or pip) ──
import importlib, subprocess, sys

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
    print("NeqSim loaded via devtools (local dev mode)")
except Exception:
    try:
        import neqsim
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
    print("NeqSim loaded via pip package")

import matplotlib.pyplot as plt
import numpy as np
import json, os, pathlib

NOTEBOOK_DIR = pathlib.Path(globals().get(
    "__vsc_ipynb_file__", os.path.abspath("step2_analysis/01_pvt_study.ipynb")
)).resolve().parent
TASK_DIR = NOTEBOOK_DIR.parent
FIGURES_DIR = TASK_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

In [ ]:
# ── Import NeqSim classes ──
if NEQSIM_MODE == "pip":
    from neqsim import jneqsim
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    SystemPrEos = jneqsim.thermo.system.SystemPrEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
    print("Classes imported from jneqsim")

## 1. Reservoir Fluid Definition

In [ ]:
# ── Define reservoir fluid ──
T_res_C = 100.0  # Reservoir temperature (degC)
P_res_bara = 300.0  # Reservoir pressure (bara)

fluid = SystemPrEos(273.15 + T_res_C, P_res_bara)

# Example gas condensate composition
fluid.addComponent("nitrogen", 0.34)
fluid.addComponent("CO2", 3.59)
fluid.addComponent("methane", 67.42)
fluid.addComponent("ethane", 9.02)
fluid.addComponent("propane", 4.31)
fluid.addComponent("i-butane", 0.93)
fluid.addComponent("n-butane", 1.71)
fluid.addComponent("i-pentane", 0.74)
fluid.addComponent("n-pentane", 0.85)
fluid.addComponent("n-hexane", 1.38)
# For C7+: use addTBPfraction or addPlusFraction
# fluid.addTBPfraction("C7", 1.0, 90.0 / 1000, 0.746)  # moles, MW, SG

fluid.setMixingRule("classic")
# fluid.setMultiPhaseCheck(True)  # Enable if liquid dropout expected

print("Fluid: {} components at {} degC, {} bara".format(
    fluid.getNumberOfComponents(), T_res_C, P_res_bara))

## 2. Saturation Pressure

In [ ]:
# ── Find saturation (bubble/dew) point ──
ops = ThermodynamicOperations(fluid)
try:
    ops.calcPTphaseEnvelope(True)  # True = experiment-like
    print("Phase envelope calculated")
except Exception as e:
    print("Phase envelope failed:", e)

# Saturation pressure at reservoir temperature
fluid_sat = fluid.clone()
fluid_sat.setTemperature(273.15 + T_res_C)
ops_sat = ThermodynamicOperations(fluid_sat)
try:
    ops_sat.bubblePointPressureFlash(False)
    P_sat = fluid_sat.getPressure()
    print("Bubble point pressure: {:.1f} bara at {:.0f} degC".format(P_sat, T_res_C))
except Exception:
    try:
        ops_sat.dewPointPressureFlash(False)
        P_sat = fluid_sat.getPressure()
        print("Dew point pressure: {:.1f} bara at {:.0f} degC".format(P_sat, T_res_C))
    except Exception as e:
        print("Saturation pressure failed:", e)
        P_sat = None

## 3. Constant Mass Expansion (CME)

In [ ]:
# ── CME at reservoir temperature ──
pressures = np.linspace(400, 50, 30)  # bara sweep
cme_results = {"P_bara": [], "rel_vol": [], "Z": [], "density_kg_m3": [],
               "liquid_vol_pct": []}

for P in pressures:
    test_fluid = fluid.clone()
    test_fluid.setPressure(float(P))
    test_fluid.setTemperature(273.15 + T_res_C)
    ops_cme = ThermodynamicOperations(test_fluid)
    ops_cme.TPflash()
    test_fluid.initProperties()

    cme_results["P_bara"].append(float(P))
    cme_results["Z"].append(test_fluid.getZ())
    cme_results["density_kg_m3"].append(test_fluid.getDensity("kg/m3"))

    n_phases = test_fluid.getNumberOfPhases()
    if n_phases > 1 and test_fluid.hasPhaseType("oil"):
        liq_frac = test_fluid.getPhase("oil").getVolume() / test_fluid.getVolume() * 100
    else:
        liq_frac = 0.0
    cme_results["liquid_vol_pct"].append(liq_frac)

print("CME: {} pressure points calculated".format(len(pressures)))

In [ ]:
# ── Plot CME results ──
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].plot(cme_results["P_bara"], cme_results["Z"], 'b-o', markersize=3)
axes[0].set_xlabel('Pressure (bara)')
axes[0].set_ylabel('Compressibility Factor Z')
axes[0].set_title('Z-factor vs Pressure')
axes[0].grid(True, alpha=0.3)

axes[1].plot(cme_results["P_bara"], cme_results["density_kg_m3"], 'r-o', markersize=3)
axes[1].set_xlabel('Pressure (bara)')
axes[1].set_ylabel('Density (kg/m\u00b3)')
axes[1].set_title('Density vs Pressure')
axes[1].grid(True, alpha=0.3)

axes[2].plot(cme_results["P_bara"], cme_results["liquid_vol_pct"], 'g-o', markersize=3)
axes[2].set_xlabel('Pressure (bara)')
axes[2].set_ylabel('Liquid Volume (%)')
axes[2].set_title('Liquid Dropout')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "cme_results.png"), dpi=150, bbox_inches="tight")
plt.show()

### Discussion

**Observation:** [CME results description with numbers]

**Physical mechanism:** [Why behavior changes at saturation pressure]

**Engineering implication:** [Impact on production/facilities]

**Recommendation:** [Actions]

## 4. Save Results

In [ ]:
results_path = TASK_DIR / "results.json"
if results_path.exists():
    with open(results_path, "r") as f:
        results = json.load(f)
else:
    results = {}

results["key_results"] = {
    "saturation_pressure_bara": P_sat if P_sat else 0.0,
    "reservoir_temperature_C": T_res_C,
    "reservoir_pressure_bara": P_res_bara,
}
results["approach"] = "PR EOS with classic mixing rule, CME at reservoir T."
results["figure_captions"] = {
    "cme_results.png": "CME results: Z-factor, density, and liquid dropout vs pressure.",
}

with open(str(results_path), "w") as f:
    json.dump(results, f, indent=2)

print("results.json saved")